# Deploying a Regression Model — Flask API and a Browser Frontend

01 Core Python · 02 Pandas · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · **▶ 09 Deployment**

`09_Model_Deployment/01_training_and_saving_a_regression_model.ipynb`

---

### In one paragraph (no jargon)

Every earlier module ends with a trained model sitting inside a notebook — useful for learning, useless to
anyone who isn't running Jupyter. This notebook closes that gap: it trains a small **Linear Regression** model
that predicts house price from area, bedrooms, bathrooms, and age, then saves it to `model.pkl` with `pickle`.
A separate Flask app, `app.py`, loads that file and exposes it as a `/predict` API; a plain HTML/JS page,
`index.html`, gives it a form to type numbers into. Together the three files are a minimal but complete
path from *trained model* to *something a non-technical person can click on*.

### Files in this module

| File | Role |
|---|---|
| `01_training_and_saving_a_regression_model.ipynb` (this notebook) | Trains the model and writes `model.pkl` |
| `app.py` | Flask app — loads `model.pkl`, serves `index.html`, and answers `POST /predict` |
| `index.html` | Form the user fills in; calls `/predict` with `fetch()` and shows the result |

Run this notebook first (it creates `model.pkl`), then start the app from a terminal in this folder with
`python app.py` and open `http://127.0.0.1:5000/` in a browser.


### Jargon buster

| Term | Plain English |
|---|---|
| Pickle (`pickle`) | Python's built-in way to save an object (like a trained model) to a file and load it back later. |
| Flask | A lightweight Python web framework — turns a Python function into a URL an app can call. |
| API endpoint | A URL that accepts a request and returns data, here JSON, instead of a full web page. |
| `fetch()` | The JavaScript function browsers use to call an API from a web page without reloading it. |


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle


### 1. Build a small synthetic housing dataset

20 houses with `area`, `bedrooms`, `bathrooms`, `age`, and the `price` we want to predict. In a real project
this would come from `datasets/`; here it is hand-built so the relationship between features and price is
clean enough to demonstrate deployment without fighting messy data.


In [2]:
data = {
    "area": [
        800, 900, 1000, 1100, 1200,
        1300, 1400, 1500, 1600, 1700,
        1800, 1900, 2000, 2100, 2200,
        2300, 2400, 2500, 2600, 2700
    ],

    "bedrooms": [
        2, 2, 2, 2, 3,
        3, 3, 3, 3, 3,
        3, 3, 4, 4, 4,
        4, 4, 4, 4, 5
    ],

    "bathrooms": [
        1, 1, 2, 2, 2,
        2, 2, 2, 3, 3,
        3, 3, 3, 3, 3,
        3, 4, 4, 4, 4
    ],

    "age": [
        20, 18, 15, 15, 12,
        10, 10, 8, 8, 7,
        6, 5, 5, 4, 4,
        3, 3, 2, 2, 1
    ],

    "price": [
        180000, 200000, 230000, 250000, 290000,
        310000, 340000, 370000, 400000, 420000,
        450000, 480000, 510000, 540000, 570000,
        600000, 630000, 660000, 700000, 750000
    ]
}

df = pd.DataFrame(data)

df


    area  bedrooms  bathrooms  age   price
0    800         2          1   20  180000
1    900         2          1   18  200000
2   1000         2          2   15  230000
3   1100         2          2   15  250000
4   1200         3          2   12  290000
5   1300         3          2   10  310000
6   1400         3          2   10  340000
7   1500         3          2    8  370000
8   1600         3          3    8  400000
9   1700         3          3    7  420000
10  1800         3          3    6  450000
11  1900         3          3    5  480000
12  2000         4          3    5  510000
13  2100         4          3    4  540000
14  2200         4          3    4  570000
15  2300         4          3    3  600000
16  2400         4          4    3  630000
17  2500         4          4    2  660000
18  2600         4          4    2  700000
19  2700         5          4    1  750000

### 2. Split features and target


In [3]:
X = df[[
    "area",
    "bedrooms",
    "bathrooms",
    "age"
]]

y = df["price"]


### 3. Train/test split

The same `train_test_split` used throughout Module 07 — 80% to train on, 20% held back to check the model
on houses it never saw.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 16
Testing samples: 4

### 4. Train the model


In [5]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained successfully!")


Model trained successfully!

### 5. Evaluate on the test set

This dataset was built with a near-linear relationship on purpose, so expect a very high R² — real housing
data will not be this clean.


In [6]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


MAE : 3587.2084730046336
RMSE: 3754.1056269146716
R²  : 0.9997117932707968

### 6. Inspect the coefficients

Each coefficient is "price change per unit increase, holding the other features fixed" — e.g. roughly
₹307 more per additional square foot of area.


In [7]:
print("Intercept:", model.intercept_)

print("\nCoefficients:")

for feature, coefficient in zip(X.columns, model.coef_):
    print(f"{feature}: {coefficient}")


Intercept: -184217.6021681755

Coefficients:
area: 307.4138791812272
bedrooms: 14052.84929748245
bathrooms: 3882.3906996650567
age: 4405.891163255093

### 7. Try a single prediction

This is exactly the shape of request `app.py` will build from the form in `index.html`: a one-row DataFrame
with the same four columns, in the same order, as training.


In [8]:
new_house = pd.DataFrame({
    "area": [2000],
    "bedrooms": [3],
    "bathrooms": [3],
    "age": [5]
})

prediction = model.predict(new_house)

print("Predicted house price:", prediction[0])


Predicted house price: 506445.3320019968

### 8. Save the model for the Flask app

`app.py` loads this exact file at startup. Run this cell before starting the Flask app for the first time,
or after retraining on new data.


In [9]:
with open("model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Model saved as model.pkl")


Model saved as model.pkl

### 9. Serve it

`model.pkl` is a generated file — it is not checked into the repository (like the other derived CSVs in
`datasets/`, it is reproducible from the notebook that made it), so run this notebook end to end at least
once before starting the app.

From a terminal, **in this folder**:

```bash
pip install flask pandas scikit-learn
python app.py
```

Then open `http://127.0.0.1:5000/` in a browser. `app.py` serves `index.html` at `/`, and the page's form
posts JSON to `/predict`, which returns `{"success": true, "prediction": <number>}` — or a `4xx`/`5xx` JSON
error if a field is missing or not a number. See `app.py` and `index.html` in this same folder for the full
implementation.
